# Session 3 — Track B: McKinsey Multi-Agent Engagement Team (Instructor Capstone)

In this capstone, students build a **multi-agent consulting engagement system** that decomposes complex client questions, delegates workstreams to specialized consulting agents, and orchestrates their collaboration using LangGraph — mirroring how a McKinsey engagement team tackles real client problems.

> **Instructor Note:** This notebook contains fully solved versions of all 4 milestones with detailed approach explanations. The self-correction loop (Milestone 4) is the most challenging — allow extra time for it.

**Engagement Scenario:** A Private Equity client is evaluating an acquisition target in the healthcare technology space. The engagement team must deliver a comprehensive due diligence assessment covering market dynamics, financial performance, competitive positioning, and operational readiness.

## Capstone Objectives

By the end of this capstone, students will have built:

1. An **Engagement Manager (supervisor) agent** that decomposes client questions into workstreams and assigns them to the right specialists
2. **Specialized consulting agents** — Strategy Analyst, Financial Analyst, Operations Expert, and Industry Researcher — each with distinct tools and expertise
3. A **LangGraph orchestration** layer managing workstream coordination, dependency sequencing, and insight synthesis
4. An **evaluation and self-correction** loop ensuring deliverable quality meets McKinsey standards

This mirrors the structure of a real McKinsey engagement: a Partner or EM coordinates workstreams, specialized analysts execute their workstreams, and the team synthesizes findings into a unified recommendation.

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
from typing import TypedDict, Annotated, List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
import operator

print("All imports successful!")

---
## Milestone 1: Engagement Manager Agent & Workstream Decomposition

Build an Engagement Manager agent that decomposes complex client questions into workstreams and assigns them to the right consulting specialists.

> **Approach:** The Engagement Manager uses a structured prompt that instructs the LLM to return JSON with workstream decomposition. Each workstream includes an ID, hypothesis to test, assigned specialist, and dependencies on other workstreams. This enables dependency-aware execution in Milestone 3 — just as in a real engagement, some analyses must complete before others can begin (e.g., market sizing before competitive positioning).

In [ ]:
# Milestone 1 — SOLUTION: Engagement Manager Agent & Workstream Decomposition

class EngagementManagerAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Engagement Manager leading a consulting engagement.
Decompose the client's question into 2-4 workstreams, each with a clear hypothesis to test.

Available specialists on your engagement team:
- strategy_analyst: Conducts market sizing, competitive analysis, and strategic positioning assessments
- financial_analyst: Performs financial modeling, valuation analysis, revenue/cost decomposition, and ROI calculations
- operations_expert: Evaluates operational readiness, process efficiency, org structure, and implementation feasibility
- industry_researcher: Researches industry trends, regulatory landscape, technology disruptions, and market dynamics

Return ONLY valid JSON in this format:
{
  "workstreams": [
    {"id": 1, "description": "...", "hypothesis": "...", "assigned_specialist": "industry_researcher", "dependencies": []},
    {"id": 2, "description": "...", "hypothesis": "...", "assigned_specialist": "strategy_analyst", "dependencies": [1]}
  ]
}"""

    def decompose(self, client_question: str) -> Dict:
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=client_question)
        ])
        try:
            plan = json.loads(response.content)
        except:
            # Fallback: simple two-workstream plan
            plan = {"workstreams": [
                {"id": 1, "description": f"Market & industry research: {client_question}", "hypothesis": "The market is attractive and growing", "assigned_specialist": "industry_researcher", "dependencies": []},
                {"id": 2, "description": f"Strategic assessment: {client_question}", "hypothesis": "The target has a defensible competitive position", "assigned_specialist": "strategy_analyst", "dependencies": [1]}
            ]}
        plan["client_question"] = client_question
        return plan


# Test — PE Due Diligence scenario
em = EngagementManagerAgent()
plan = em.decompose(
    "Our PE client is considering acquiring a mid-market healthcare IT company. "
    "Assess the target's market position, financial health, operational scalability, "
    "and provide a go/no-go recommendation with key risks and value creation levers."
)
print(json.dumps(plan, indent=2))

---
## Milestone 2: Specialized Consulting Agents

Build specialized consulting agents, each with unique system prompts and analytical capabilities. These mirror the roles on a real McKinsey engagement team.

> **Approach:** Each specialist has a carefully crafted system prompt that defines their consulting expertise, analytical frameworks, and output format. Specialists receive both their workstream description and context from previously completed workstreams, allowing them to build on each other's insights — just as analysts on an engagement share findings across workstreams.

In [ ]:
# Milestone 2 — SOLUTION: Specialized Consulting Agents

class StrategyAnalystAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Strategy Analyst specializing in competitive analysis and market positioning.
Given a workstream assignment:
1. Define the relevant market and key segments
2. Identify the competitive landscape (key players, market shares, positioning)
3. Assess the target's competitive advantages and vulnerabilities
4. Develop a hypothesis on strategic positioning with supporting evidence
5. Use frameworks like Porter's Five Forces, value chain analysis, or growth-share matrix where relevant

Structure your output with clear headers, bullet points, and a summary of key insights.
Use precise consulting language: hypotheses, implications, so-whats, and recommendations."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "strategy_analyst"}


class FinancialAnalystAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Financial Analyst specializing in financial modeling and valuation.
Given a workstream assignment:
1. Analyze revenue drivers and cost structure (decompose into fixed/variable, direct/indirect)
2. Assess margin trajectory and unit economics
3. Evaluate cash flow generation and working capital efficiency
4. Identify key financial risks and value creation levers
5. Provide a preliminary view on valuation (multiples-based or DCF framework)

Structure your output with clear financial metrics, benchmarks against industry peers, and actionable insights.
Quantify everything — use ranges where exact figures are unavailable."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "financial_analyst"}


class OperationsExpertAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Operations Expert specializing in operational assessment and transformation.
Given a workstream assignment:
1. Evaluate the operating model (org structure, processes, technology stack)
2. Assess scalability — can the organization grow 2-3x without breaking?
3. Identify operational risks and bottlenecks
4. Propose efficiency improvements and implementation roadmap
5. Estimate the effort, timeline, and investment required for key initiatives

Structure your output with clear categories, maturity assessments, and prioritized recommendations.
Use frameworks like operating model canvas, lean assessment, or capability maturity models."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "operations_expert"}


class IndustryResearcherAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Industry Researcher specializing in sector-specific knowledge and market intelligence.
Given a workstream assignment:
1. Map the industry landscape: market size, growth rate, key trends
2. Identify regulatory factors, compliance requirements, and policy tailwinds/headwinds
3. Assess technology disruptions and their impact on the sector
4. Analyze customer segments, buying behaviors, and unmet needs
5. Provide forward-looking perspective: where is this industry heading in 3-5 years?

Structure your output with data-driven insights, sourced estimates where possible, and clear implications.
Be rigorous in separating facts from hypotheses."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "industry_researcher"}


# Test — Strategy Analyst on a competitive analysis workstream
strategy_analyst = StrategyAnalystAgent()
result = strategy_analyst.execute(
    "Assess the competitive positioning of the target healthcare IT company. "
    "Identify top 3-5 competitors, relative market share, and key differentiators."
)
print(f"Specialist: {result['worker']}")
print(f"Analysis (preview): {result['result'][:400]}...")

---
## Milestone 3: LangGraph Engagement Orchestration

Build a LangGraph workflow that orchestrates the Engagement Manager and consulting specialists into a coordinated engagement team.

> **Approach:** We define a StateGraph with an Engagement Manager node (creates the workstream plan), a dispatcher node (picks the next workstream and routes it to the right specialist), specialist nodes (execute workstreams), and a synthesis node (combines all workstream insights into a unified client deliverable). The dispatcher uses conditional routing to assign each workstream to the correct specialist, and loops back for more workstreams until the full engagement plan is executed — mirroring how an EM coordinates a real engagement.

In [ ]:
# Milestone 3 — SOLUTION: LangGraph Engagement Orchestration

class EngagementState(TypedDict):
    client_question: str
    plan: Dict
    completed_workstreams: List[Dict]
    current_workstream_idx: int
    deliverable: str

# Initialize the engagement team
em = EngagementManagerAgent()
specialists = {
    "strategy_analyst": StrategyAnalystAgent(),
    "financial_analyst": FinancialAnalystAgent(),
    "operations_expert": OperationsExpertAgent(),
    "industry_researcher": IndustryResearcherAgent()
}

def em_node(state: EngagementState) -> EngagementState:
    """Engagement Manager creates the workstream plan."""
    plan = em.decompose(state["client_question"])
    return {**state, "plan": plan, "completed_workstreams": [], "current_workstream_idx": 0}

def dispatcher_node(state: EngagementState) -> EngagementState:
    """Determine the next workstream to execute."""
    return state  # Just pass through — routing logic is in the conditional edge

def specialist_node(state: EngagementState) -> EngagementState:
    """Execute the current workstream with the assigned specialist."""
    workstreams = state["plan"]["workstreams"]
    idx = state["current_workstream_idx"]
    workstream = workstreams[idx]
    
    # Build context from completed workstreams (cross-workstream insight sharing)
    context = "\n\n".join([
        f"[{cw['worker'].upper()}] {cw['result'][:300]}" for cw in state["completed_workstreams"]
    ])
    
    # Execute with the assigned specialist
    specialist_type = workstream["assigned_specialist"]
    specialist = specialists.get(specialist_type, specialists["strategy_analyst"])
    result = specialist.execute(workstream["description"], context)
    
    completed = state["completed_workstreams"] + [{**result, "workstream_id": workstream["id"]}]
    return {**state, "completed_workstreams": completed, "current_workstream_idx": idx + 1}

def synthesis_node(state: EngagementState) -> EngagementState:
    """Synthesize all workstream outputs into a unified client deliverable."""
    parts = []
    for cw in state["completed_workstreams"]:
        specialist_title = cw["worker"].replace("_", " ").title()
        parts.append(f"--- [{specialist_title} Workstream] ---\n{cw['result']}")
    deliverable = "\n\n".join(parts)
    return {**state, "deliverable": deliverable}

def route_dispatcher(state: EngagementState) -> str:
    """Route to next specialist or synthesis based on remaining workstreams."""
    workstreams = state["plan"].get("workstreams", [])
    if state["current_workstream_idx"] < len(workstreams):
        return "specialist"
    return "synthesis"

# Build the engagement graph
graph = StateGraph(EngagementState)
graph.add_node("engagement_manager", em_node)
graph.add_node("dispatcher", dispatcher_node)
graph.add_node("specialist", specialist_node)
graph.add_node("synthesis", synthesis_node)

graph.set_entry_point("engagement_manager")
graph.add_edge("engagement_manager", "dispatcher")
graph.add_conditional_edges("dispatcher", route_dispatcher, {"specialist": "specialist", "synthesis": "synthesis"})
graph.add_edge("specialist", "dispatcher")
graph.add_edge("synthesis", END)

engagement_app = graph.compile()

# Test — Market entry strategy engagement
result = engagement_app.invoke({
    "client_question": (
        "A Fortune 500 industrial company wants to enter the Southeast Asian market "
        "for smart manufacturing solutions. Assess market attractiveness, competitive "
        "landscape, financial feasibility, and recommend an entry strategy."
    ),
    "plan": {}, "completed_workstreams": [], "current_workstream_idx": 0, "deliverable": ""
})

print(f"Completed {len(result['completed_workstreams'])} workstreams")
print(f"\nClient deliverable preview:\n{result['deliverable'][:500]}...")

---
## Milestone 4: Partner Review & Quality Assurance Loop

Add a Partner-level quality review and self-correction loop to ensure deliverables meet McKinsey standards before going to the client.

> **Approach:** We add a Partner Review node that evaluates the synthesized deliverable on three dimensions: analytical rigor, actionability of recommendations, and strategic coherence. If the average score falls below the quality threshold (3.5/5), a revision node identifies specific weaknesses and rewrites. We track iteration count and cap retries at 2 — just as in a real engagement, there is a limit on revision cycles before the final deliverable must ship.

In [ ]:
# Milestone 4 — SOLUTION: Partner Review & Quality Assurance Loop

class EnhancedEngagementState(TypedDict):
    client_question: str
    plan: Dict
    completed_workstreams: List[Dict]
    current_workstream_idx: int
    deliverable: str
    scores: List[Dict]
    revision_count: int

partner_llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def partner_review_node(state: EnhancedEngagementState) -> EnhancedEngagementState:
    """Partner reviews the deliverable for McKinsey-quality standards."""
    metrics = {}
    for metric, prompt in {
        "analytical_rigor": (
            f"Rate 1-5: Does this deliverable demonstrate rigorous, structured analysis with "
            f"clear hypotheses, supporting evidence, and logical reasoning?\n"
            f"Client question: {state['client_question']}\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        ),
        "actionability": (
            f"Rate 1-5: Are the recommendations specific, actionable, and prioritized? "
            f"Could a CEO act on these immediately?\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        ),
        "strategic_coherence": (
            f"Rate 1-5: Do the different workstream outputs tell a coherent story? "
            f"Are the insights synthesized into a unified strategic narrative?\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        )
    }.items():
        resp = partner_llm.invoke([
            SystemMessage(content="You are a McKinsey Senior Partner reviewing a client deliverable. Return ONLY a number 1-5."),
            HumanMessage(content=prompt)
        ])
        try:
            metrics[metric] = int(resp.content.strip())
        except:
            metrics[metric] = 3
    
    metrics["average"] = round(sum(metrics.values()) / len(metrics), 2)
    scores = state.get("scores", []) + [metrics]
    print(f"  Partner Review (iteration {len(scores)}): {metrics}")
    return {**state, "scores": scores}

def revision_node(state: EnhancedEngagementState) -> EnhancedEngagementState:
    """Revise the deliverable based on Partner feedback — focus on the weakest dimension."""
    latest_scores = state["scores"][-1]
    
    # Identify weakest area
    weakest = min(
        ["analytical_rigor", "actionability", "strategic_coherence"],
        key=lambda k: latest_scores[k]
    )
    
    weakness_labels = {
        "analytical_rigor": "analytical rigor — strengthen hypotheses, add supporting evidence, sharpen logic",
        "actionability": "actionability — make recommendations more specific, prioritized, and implementation-ready",
        "strategic_coherence": "strategic coherence — better synthesize workstream insights into a unified narrative"
    }
    
    response = partner_llm.invoke([
        SystemMessage(content=f"""You are a McKinsey Engagement Manager revising a client deliverable based on Partner feedback.
The Partner flagged {weakness_labels[weakest]} as the key area for improvement.
Rewrite the deliverable to address this feedback while preserving the strong elements.
Maintain McKinsey-quality structure: situation, complication, resolution; clear exhibits; and a strong 'so what'."""),
        HumanMessage(content=f"Client question: {state['client_question']}\n\nCurrent deliverable:\n{state['deliverable']}")
    ])
    
    revision_count = state.get("revision_count", 0) + 1
    print(f"  Revised deliverable (focus: {weakest}, iteration {revision_count})")
    return {**state, "deliverable": response.content, "revision_count": revision_count}

def route_partner_review(state: EnhancedEngagementState) -> str:
    """Route based on Partner quality score and revision count."""
    scores = state.get("scores", [])
    revision_count = state.get("revision_count", 0)
    
    if not scores:
        return "end"
    
    latest = scores[-1]["average"]
    if latest >= 3.5 or revision_count >= 2:
        return "end"
    return "revise"

# Build enhanced engagement graph with Partner review loop
enhanced_graph = StateGraph(EnhancedEngagementState)
enhanced_graph.add_node("engagement_manager", em_node)
enhanced_graph.add_node("dispatcher", dispatcher_node)
enhanced_graph.add_node("specialist", specialist_node)
enhanced_graph.add_node("synthesis", synthesis_node)
enhanced_graph.add_node("partner_review", partner_review_node)
enhanced_graph.add_node("reviser", revision_node)

enhanced_graph.set_entry_point("engagement_manager")
enhanced_graph.add_edge("engagement_manager", "dispatcher")
enhanced_graph.add_conditional_edges("dispatcher", route_dispatcher, {"specialist": "specialist", "synthesis": "synthesis"})
enhanced_graph.add_edge("specialist", "dispatcher")
enhanced_graph.add_edge("synthesis", "partner_review")
enhanced_graph.add_conditional_edges("partner_review", route_partner_review, {"revise": "reviser", "end": END})
enhanced_graph.add_edge("reviser", "partner_review")

enhanced_engagement_app = enhanced_graph.compile()

# Test — Full PE due diligence engagement with Partner review
result = enhanced_engagement_app.invoke({
    "client_question": (
        "Our PE client is evaluating a $500M acquisition of a healthcare IT platform company. "
        "Conduct a comprehensive due diligence covering market dynamics, competitive positioning, "
        "financial health, and operational scalability. Provide a go/no-go recommendation "
        "with key risks, value creation levers, and a 100-day integration plan."
    ),
    "plan": {}, "completed_workstreams": [], "current_workstream_idx": 0,
    "deliverable": "", "scores": [], "revision_count": 0
})

print(f"\nCompleted {len(result['completed_workstreams'])} workstreams, {result['revision_count']} revisions")
print(f"Partner review scores: {[s['average'] for s in result['scores']]}")
print(f"\nFinal client deliverable:\n{result['deliverable'][:600]}...")

---
## Capstone Summary

Students built a **McKinsey Multi-Agent Engagement Team** — a production-ready multi-agent system modeled after real consulting engagement structure:

1. **Engagement Manager (Supervisor)** -- Decomposes client questions into workstreams with hypotheses, assigns specialists, and manages dependencies
2. **Consulting Specialists (Workers)** -- Strategy Analyst, Financial Analyst, Operations Expert, and Industry Researcher — each with domain-specific analytical frameworks
3. **Engagement Orchestration** -- LangGraph workflow with conditional routing, cross-workstream context sharing, and deliverable synthesis
4. **Partner Review (Self-Correction)** -- Quality evaluation on analytical rigor, actionability, and strategic coherence — with automated revision until McKinsey standards are met

**Key consulting scenarios demonstrated:** PE due diligence, market entry strategy, digital transformation assessment, cost optimization programs.

**Up Next:** Session 4 -- Cross-track presentations, AI governance, and closing.